# Geração do dataset de vendas (fictício)

> ⚠️ **Dados fictícios.** O dataset é gerado em Python (notebook `geracao_dataset.ipynb`). Sazonalidade, variação entre meses, distribuição por estado e base de clientes foram **simuladas**.

Rode esta célula **uma vez**: ela cria o arquivo `vendas_raw.csv`, lido pelo notebook `01_tratamento_eda.ipynb`.

In [1]:
import random
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

# Seeds: garantem que os dados gerados sejam sempre os mesmos
random.seed(42)
np.random.seed(42)


def dsa_gera_dados_ficticios(pedidos_base_dia = 900, dias = 730):

    """
    Gera um DataFrame do Pandas com dados de vendas fictícias (2026-2027),
    com sazonalidade, variações por mês/ano, concentração geográfica e ruído SIMULADOS.
    """

    print(f"\nIniciando a geração de vendas ({dias} dias, base de {pedidos_base_dia} pedidos/dia)...")

    # Dicionário com produtos, suas categorias e preços
    produtos = {
        'Laptop Gamer': {'categoria': 'Eletrônicos', 'preco': 7500.00},
        'Mouse Vertical': {'categoria': 'Acessórios', 'preco': 250.00},
        'Teclado Mecânico': {'categoria': 'Acessórios', 'preco': 550.00},
        'Monitor Ultrawide': {'categoria': 'Eletrônicos', 'preco': 2800.00},
        'Cadeira Gamer': {'categoria': 'Móveis', 'preco': 1200.00},
        'Headset 7.1': {'categoria': 'Acessórios', 'preco': 800.00},
        'Placa de Vídeo': {'categoria': 'Hardware', 'preco': 4500.00},
        'SSD 1TB': {'categoria': 'Hardware', 'preco': 600.00}
    }
    lista_produtos = list(produtos.keys())

    # Dicionário com cidades e seus respectivos estados
    cidades_estados = {
        'São Paulo': 'SP', 'Rio de Janeiro': 'RJ', 'Belo Horizonte': 'MG',
        'Porto Alegre': 'RS', 'Salvador': 'BA', 'Curitiba': 'PR', 'Fortaleza': 'CE'
    }
    lista_cidades = list(cidades_estados.keys())

    # Peso de cada cidade nos pedidos (concentração no Sudeste, SP em 1º)
    pesos_cidades = {'São Paulo': 35, 'Rio de Janeiro': 18, 'Belo Horizonte': 15,
                     'Curitiba': 10, 'Porto Alegre': 9, 'Salvador': 8, 'Fortaleza': 5}
    lista_pesos_cidades = [pesos_cidades[c] for c in lista_cidades]

    # --- Parâmetros da sazonalidade simulada ---
    # Fator de volume por mês (1.00 = normal): sazonalidade que se repete todo ano
    fator_mes = {1: 0.90, 2: 0.85, 3: 0.95, 4: 0.95, 5: 1.05, 6: 0.95,
                 7: 0.95, 8: 0.95, 9: 0.95, 10: 1.00, 11: 1.25, 12: 1.30}

    # Fator por dia da semana (0 = segunda ... 6 = domingo)
    fator_semana = {0: 1.05, 1: 1.05, 2: 1.00, 3: 1.00, 4: 1.05, 5: 0.90, 6: 0.85}

    # Ajuste específico de cada mês/ano (campanhas, rupturas, etc.):
    # faz alguns meses de 2026 serem melhores que os de 2027, e vice-versa
    ajuste_ano_mes = {
        (2026, 1): 1.00, (2026, 2): 1.05, (2026, 3): 1.15, (2026, 4): 1.00,
        (2026, 5): 0.98, (2026, 6): 1.15, (2026, 7): 0.95, (2026, 8): 1.12,
        (2026, 9): 0.95, (2026, 10): 1.12, (2026, 11): 0.95, (2026, 12): 0.97,
        (2027, 1): 1.05, (2027, 2): 1.10, (2027, 3): 0.92, (2027, 4): 1.10,
        (2027, 5): 1.12, (2027, 6): 0.90, (2027, 7): 1.15, (2027, 8): 0.90,
        (2027, 9): 1.12, (2027, 10): 0.95, (2027, 11): 1.15, (2027, 12): 1.12,
    }

    crescimento_anual = 0.04   # crescimento gradual leve (~4% ao ano)
    ruido_dia = 0.07           # variação aleatória diária (desvio padrão de 7%)

    data_inicial = datetime(2026, 1, 1)
    datas = [data_inicial + timedelta(days=d) for d in range(dias)]

    # Black Friday = última sexta-feira de novembro de cada ano
    def black_friday(ano):
        d = datetime(ano, 11, 30)
        while d.weekday() != 4:
            d -= timedelta(days=1)
        return d
    bf_datas = {black_friday(a) for a in {x.year for x in datas}}

    # Lista que armazenará os registros de vendas
    dados_vendas = []
    id_pedido = 1000

    # Loop por dia: define quantos pedidos o dia terá e gera cada um
    for d, data_dia in enumerate(datas):

        tendencia = 1 + crescimento_anual * (d / 365)
        fator = (fator_mes[data_dia.month] * fator_semana[data_dia.weekday()]
                 * tendencia * ajuste_ano_mes[(data_dia.year, data_dia.month)])
        fator *= random.gauss(1, ruido_dia)

        # Pico na Black Friday e nos dias vizinhos
        if data_dia in bf_datas:
            fator *= 2.5
        elif (data_dia + timedelta(days=1)) in bf_datas or (data_dia - timedelta(days=1)) in bf_datas:
            fator *= 1.5

        n_pedidos = max(1, int(round(pedidos_base_dia * fator)))

        # Em novembro e dezembro, Eletrônicos e Acessórios vendem mais (presentes)
        if data_dia.month in (11, 12):
            pesos = [1.3 if produtos[p]['categoria'] in ('Eletrônicos', 'Acessórios') else 1.0
                     for p in lista_produtos]
        else:
            pesos = [1.0] * len(lista_produtos)

        for _ in range(n_pedidos):

            # Seleciona produto (com pesos), cidade (com pesos) e quantidade
            produto_nome = random.choices(lista_produtos, weights=pesos)[0]
            cidade = random.choices(lista_cidades, weights=lista_pesos_cidades)[0]
            quantidade = np.random.randint(1, 8)

            # Data do pedido: dia atual + hora aleatória
            data_pedido = data_dia + timedelta(hours = random.randint(0, 23))

            # Se o produto for Mouse ou Teclado, aplica desconto aleatório de até 10%
            if produto_nome in ['Mouse Vertical', 'Teclado Mecânico']:
                preco_unitario = produtos[produto_nome]['preco'] * np.random.uniform(0.9, 1.0)
            else:
                preco_unitario = produtos[produto_nome]['preco']

            dados_vendas.append({
                'ID_Pedido': id_pedido,
                'Data_Pedido': data_pedido,
                'Nome_Produto': produto_nome,
                'Categoria': produtos[produto_nome]['categoria'],
                'Preco_Unitario': round(preco_unitario, 2),
                'Quantidade': quantidade,
                'ID_Cliente': np.random.randint(1000, 151000),   # ~150 mil clientes
                'Cidade': cidade,
                'Estado': cidades_estados[cidade]
            })
            id_pedido += 1

    print("Geração de dados concluída.\n")
    return pd.DataFrame(dados_vendas)


# Gera o DataFrame e salva o arquivo bruto na pasta do notebook
df = dsa_gera_dados_ficticios()
df.to_csv("vendas_raw.csv", index=False)

# Validações
print("Shape:", df.shape)                               # esperado: ~ (707161, 9)
print("Primeira data:", df["Data_Pedido"].min())        # esperado: 2026-01-01
print("Última data:", df["Data_Pedido"].max())          # esperado: 2027-12-31
print("IDs únicos:", df["ID_Pedido"].nunique())         # igual ao número de linhas
print("Clientes únicos:", df["ID_Cliente"].nunique())   # esperado: ~148 mil

# Participação de cada estado no faturamento (conferência da concentração no Sudeste)
fat_uf = (df["Preco_Unitario"] * df["Quantidade"]).groupby(df["Estado"]).sum()
print((fat_uf / fat_uf.sum() * 100).round(1).sort_values(ascending=False))
df.head()


Iniciando a geração de vendas (730 dias, base de 900 pedidos/dia)...
Geração de dados concluída.

Shape: (707161, 9)
Primeira data: 2026-01-01 00:00:00
Última data: 2027-12-31 23:00:00
IDs únicos: 707161
Clientes únicos: 148653
Estado
SP    35.1
RJ    17.9
MG    15.1
PR     9.9
RS     8.9
BA     8.0
CE     5.0
dtype: float64


,ID_Pedido,Data_Pedido,Nome_Produto,Categoria,Preco_Unitario,Quantidade,ID_Cliente,Cidade,Estado
0,1000,2026-01-01 23:00:00,Teclado Mecânico,Acessórios,538.81,7,104694,São Paulo,SP
1,1001,2026-01-01 17:00:00,Laptop Gamer,Eletrônicos,7500.00,3,120879,Porto Alegre,RS
2,1002,2026-01-01 00:00:00,Laptop Gamer,Eletrônicos,7500.00,5,55886,Rio de Janeiro,RJ
3,1003,2026-01-01 19:00:00,Laptop Gamer,Eletrônicos,7500.00,2,88498,São Paulo,SP
4,1004,2026-01-01 20:00:00,Laptop Gamer,Eletrônicos,7500.00,5,17023,São Paulo,SP
